In [3]:
pip install chess

  Using cached chess-1.11.2.tar.gz (6.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147877 sha256=784e1d3c9daf19f4d0438f7c2868746eca1b62dab94c82f76790f33dcdea4c25
  Stored in directory: /Users/kraussry/Library/Caches/pip/wheels/06/53/ec/27b065e5de042efb4c622f661c996599af695bff1b9db4163b
Successfully built chess
Note: you may need to restart the kernel to use updated packages.


In [4]:
import chess

In [5]:
import glob

In [6]:
import re

## Goal

Read in an annotated pgn from lichess, process it, and output pdf flashcards from each relevant position.


- ignore the standard lichess annotations and eval statements
    - clean the comments before deciding if a position should be added to the pdf output
 
### Step 1: Lichess PGN to markdown or something else

- is there value in markdown flashcard form?
    - it is different than my initial FEN flashcard form, but it is a recognized standard

### Step 2: markdown or something else to PDF via latex

- if there is no answer, the markdown approach kind of doesn't work
    - if there isn't an answer provided, I am saying it is obvious, I guess....
 
### Questions

- if I put a ? on a line by itself in the comment, should that mean the standard markdown flashcard form of question above and anser below?
- should I output a standard text file so that generating the pdf is handled separately?
    - I could store the text output as a list and pass it through that other code at the end
- how do I ensure correct orientation?
    - will I sometimes put the comments on the wrong move?

In [30]:
url = "https://lichess.org/study/jUNlB2KL.pgn"

In [31]:
import requests

In [32]:
response = requests.get(url)

In [33]:
response

<Response [200]>

In [34]:
print(response.text)

[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.03.03"]
[White "ryan49428"]
[Black "Radi_Dobreva"]
[Result "0-1"]
[WhiteElo "1355"]
[BlackElo "1299"]
[TimeControl "900+10"]
[Termination "Unknown"]
[Annotator "lichess.org"]
[GameId "RcFu4HI9"]
[Variant "Standard"]
[ECO "C44"]
[Opening "Scotch Game: Scotch Gambit, London Defense"]
[StudyName "March 2026 Games"]
[ChapterName "Scan the board!!!!!!!!"]
[ChapterURL "https://lichess.org/study/jUNlB2KL/RBzj3ZR2"]

1. e4 e5 2. Nf3 Nc6 3. d4 exd4 4. Bc4 Bb4+ { C44 Scotch Game: Scotch Gambit, London Defense } 5. c3 dxc3 6. O-O Nf6 7. bxc3 Bc5 8. e5 Ne4?? { (-0.08 → 4.08) Blunder. d5 was best. } (8... d5 9. exf6 dxc4 10. fxg7 Rg8 11. Nbd2 Rxg7 12. Qe2+ Qe7 13. Ne4 Bg4 14. h3) 9. Bxf7+?? { (4.08 → 1.79) Blunder. Qd5 was best. } (9. Qd5 O-O 10. Qxe4 d6 11. Bg5 Qe8 12. Bd3 f5 13. exf6 Qxe4 14. Bxe4 gxf6) 9... Kxf7 10. Qd5+ Kf8 11. Qxe4 d5 12. Qf4+ Ke8 13. Ng5?! { (1.38 → 0.43) Inaccuracy. Nbd2 was best. } (13. Nbd2 Rf8 14. Qg3 Kf7 15. Nb3 Be7 16.

In [35]:
fn = "dowloaded_pgn.pgn"

In [36]:
f = open(fn,'w')

In [37]:
f.write(response.text)

1948

In [38]:
f.close()

In [39]:
black_template = r"""```chessboard
fen: %s
orientation: black
```"""
black_template

'```chessboard\nfen: %s\norientation: black\n```'

In [40]:
white_template = r"""```chessboard
fen: %s
orientation: white
```"""
white_template

'```chessboard\nfen: %s\norientation: white\n```'

In [41]:
pgn_files = glob.glob("*.pgn")

In [42]:
pgn_files

['lichess_study_my-accelerated-dragon-games-as-black_interesting-opposite-side-castling-game_by_ryanGT_2026.02.08.pgn',
 'lichess_study_my-accelerated-dragon-games-as-black_important-opening-mistakes_by_ryanGT_2025.11.02.pgn',
 'lichess_study_my-accelerated-dragon-games-as-black_by_ryanGT_2025.11.02.pgn',
 'lichess_study_my-accelerated-dragon-games-as-black_interesting-opposite-side-castling-game_by_ryanGT_2026.02.08 (1).pgn',
 'dowloaded_pgn.pgn']

In [43]:
#fn = pgn_files[0]
#fn

In [44]:
import chess.pgn

In [45]:
pgn = open(fn)

In [46]:
first_game = chess.pgn.read_game(pgn)

In [47]:
my_user_names = ['ryan49428','ryanGT','ryanGT_e4']

In [48]:
def find_my_side(game):
    white_user = game.headers["White"]
    if white_user in my_user_names:
        return "white"
    black_user = game.headers["Black"]
    if black_user in my_user_names:
        return "black"

In [49]:
first_game.headers["White"]

'ryan49428'

In [50]:
first_game.headers["Black"]

'Radi_Dobreva'

In [51]:
myside = find_my_side(first_game)
myside

'white'

In [52]:
board = first_game.board()

In [53]:
get_rid_of_eval = re.compile(r'\[.*eval.*?\]')

In [54]:
std_comment = re.compile(r"(Inaccuracy|Blunder|Mistake).*?[was best]\.")

In [55]:
#best_move_1 = re.compile("[BNRQK]*[a-h][1-8]")
best_move_1 = re.compile(r"[BNRQK][a-h][1-8] was best\.")
best_move_2 = re.compile(r"[a-h][1-8] was best\.")

In [56]:
def eliminate_best_move(movein):
    out1 = best_move_1.sub("", movein)
    out2 = best_move_2.sub("",out1)
    return out2

In [57]:
def find_non_standard_comment(line):
    # does the comment contain anything not auto-generated by lichess/stockfish?
    # Inaccuracy|Blunder|Mistake. e5 was best.
    # Checkmate is now unavoidable.
    out1 = get_rid_of_eval.sub('', line)
    out1 = out1.strip()
    out2 = std_comment.sub("",out1)
    out2 = out2.strip()
    auto_phrases = ["Checkmate is now unavoidable.", "0-1 Black wins.", "1-0 White wins."]
    out3 = out2
    for phrase in auto_phrases:
        out3 = out3.replace(phrase, "")
    out4 = eliminate_best_move(out3)
    #"[a-h][1-8]"
    #Checkmate is now unavoidable. Rh2 was best. 0-1 Black wins.
    return out4

### Break Comments if '?' on line by itself

In [58]:
def break_comments(comment):
    answer = ''
    question = comment
    if '\n' in comment:
        mylist = comment.split('\n')
        for i, line in enumerate(mylist):
            if line.strip() == '?':
                ind = i
                question = '\n'.join(mylist[0:ind])
                answer = '\n'.join(mylist[ind+1:])
                break
    return question, answer

In [59]:
out_list = []

In [60]:
out = out_list.append

In [61]:
extend = out_list.extend

In [62]:
if myside == "white":
    template = white_template
elif myside == "black":
    template = black_template
else:
    raise ValueError("myside not defined")

In [63]:
template

'```chessboard\nfen: %s\norientation: white\n```'

In [64]:
board = first_game.board()
for node in first_game.mainline():
    # Print the move and its associated comment
    #print(f"Move: {node.move}, Comment: {node.comment}")
    cur_com = node.comment
    clean_com = find_non_standard_comment(cur_com).strip()
    if clean_com:
        print(clean_com)
        q,a = break_comments(clean_com)
        if not a:
            a = "no answer given"
        q = q.replace("%","\\%")
        a = a.replace("%","\\%")
        #print("question: %s" % q)
        #print("answer: %s" % a)
        out(template % board.fen())
        out(q)
        out("?")
        out(a)
        out("")
    board.push(node.move)
    # Check for specifically formatted evaluation comments
    #if node.comment and "[%eval" in node.comment:
    #    print(f"Evaluation found: {node.comment}")

C44 Scotch Game: Scotch Gambit, London Defense
(-0.08 → 4.08)
(4.08 → 1.79)
(1.38 → 0.43)
(0.85 → 3.70)
(3.72 → 5.84)
(5.58 → 7.91)
(7.73 → Mate in 10)  Bx
(Mate in 10 → Mate in 1) Lost forced checkmate sequence. Nxc7+ was best. Scan the board!!! Why did my opponent make that move?
Black wins.


In [282]:
outname = "md_flash_card_test.md"

In [283]:
from krauss_misc import txt_mixin

In [284]:
txt_mixin.dump(outname, out_list)

In [285]:
pwd

'/Users/kraussry/Work_vault_ios/chess/Analyzing_my_games/lichess_pgn_processing'

## Stuff I tried for learning

In [189]:
re.search(r'\[.*eval.*?\]', cur_com)

<re.Match object; span=(0, 11), match='[%eval #-1]'>

In [190]:
get_rid_of_eval = re.compile(r'\[.*eval.*?\]')

In [191]:
get_rid_of_eval.sub('', cur_com)

' Checkmate is now unavoidable. Rh2 was best. 0-1 Black wins.'

In [192]:
test2 = "[%eval -0.81] Inaccuracy. e5 was best."

In [193]:
std_com = re.compile(r"(Inaccuracy|Blunder|Mistake).*?[was best]\.")

In [194]:
out1 = std_com.sub("",test2)

In [195]:
out1

'[%eval -0.81] '

In [196]:
get_rid_of_eval.sub('', out1)

' '

In [197]:
test2

'[%eval -0.81] Inaccuracy. e5 was best.'